# LLM Applications: Text Classification, Summarization & Log Analysis

This notebook demonstrates the LLM pipeline in `scripts/llm/`:
- Zero-shot and few-shot text classification
- Document summarization (single-pass and map-reduce)
- Log triage, incident summarization, and root-cause analysis

> **Prerequisites**: Set the `OPENAI_API_KEY` environment variable before running cells that call the OpenAI API.

In [ ]:
# !pip install -r ../requirements.txt

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'  # Set your key here or in the environment

In [ ]:
import sys
sys.path.insert(0, '..')

from scripts.llm.text_classification import TextClassifier
from scripts.llm.text_summarization import TextSummarizer
from scripts.llm.log_analysis import LogAnalyzer

## 1. Text Classification

Zero-shot classification of support tickets into predefined categories.

In [ ]:
LABELS = ['Billing', 'Technical Support', 'General Inquiry', 'Complaint']
EXAMPLES = [
    {'text': 'I was charged twice for my subscription.', 'label': 'Billing'},
    {'text': 'My app crashes every time I open it.',     'label': 'Technical Support'},
    {'text': 'What are your opening hours?',             'label': 'General Inquiry'},
]

classifier = TextClassifier(labels=LABELS, examples=EXAMPLES)

In [ ]:
tickets = [
    'The invoice shows the wrong amount.',
    'I cannot log in to my account after the update.',
    'Where can I find your refund policy?',
    'This service is absolutely terrible!',
]

for ticket in tickets:
    label = classifier.classify(ticket)
    print(f'[{label}] {ticket}')

## 2. Text Summarization

Summarise a long-form article using the map-reduce strategy.

In [ ]:
ARTICLE = """
Artificial intelligence (AI) has rapidly evolved over the past decade, transforming industries
ranging from healthcare and finance to transportation and entertainment. Machine learning models
can now diagnose diseases from medical images, predict stock market movements, power autonomous
vehicles, and generate realistic synthetic media.

However, these advances also raise significant concerns around algorithmic bias, data privacy,
and the displacement of human labour. Researchers and policymakers are increasingly calling for
ethical frameworks, transparency requirements, and regulation to ensure that AI systems are
developed and deployed responsibly.

Large language models (LLMs) such as GPT-4 have demonstrated remarkable few-shot and zero-shot
learning capabilities, enabling applications that previously required large labelled datasets.
This democratisation of AI is accelerating innovation but also amplifying risks related to
misinformation, intellectual property, and the concentration of AI power in a few corporations.
"""

summarizer = TextSummarizer(style='bullet-point', max_words=80)
summary = summarizer.summarise(ARTICLE)
print('Summary:\n', summary)

## 3. Log Analysis & Incident Summarization

In [ ]:
SAMPLE_LOGS = [
    'Jan 15 03:14:07 webserver01 nginx[4512]: ERROR 502 Bad Gateway upstream timed out (110: Connection timed out)',
    'Jan 15 03:14:09 webserver01 nginx[4512]: ERROR 502 Bad Gateway upstream timed out (110: Connection timed out)',
    'Jan 15 03:14:10 dbserver01 postgres[7890]: FATAL terminating connection due to administrator command',
    'Jan 15 03:14:11 dbserver01 postgres[7890]: FATAL the database system is shutting down',
    'Jan 15 03:14:15 webserver01 app[3301]: ERROR Database connection pool exhausted – all 100 connections in use',
    'Jan 15 03:14:20 monitor01 alertmanager: CRITICAL service=api latency_p99=4500ms threshold=2000ms',
    'Jan 15 03:14:22 webserver01 app[3301]: ERROR Failed to process request: timeout after 5000ms',
]

In [ ]:
analyzer = LogAnalyzer()

print('=== TRIAGE ===')
print(analyzer.triage(SAMPLE_LOGS, top_n=3))

In [ ]:
print('=== INCIDENT SUMMARY ===')
summary = analyzer.summarise_incident(SAMPLE_LOGS)
print(summary)

In [ ]:
print('=== ROOT CAUSE ANALYSIS ===')
print(analyzer.analyse_root_cause(summary))